In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm
import matplotlib.ticker as ticker

from f_compare_bn import * 
from f_recog_prior_norm_plot import plot_z_white_fig4_style

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # hes or bs
barr_type = 'barr' # van or barr
opt_type = 'call' # call or put
chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

if not(opt_type == 'call' or  opt_type == 'put'):
    raise ValueError("option_type must be 'call' or 'put'")

if not(barr_type == 'van' or  barr_type == 'barr'):
    raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.115733
        else: # put
            bench_price = 0.005170
    else: # van
        if opt_type == 'call':
            bench_price = 0.124491
        else: # put
            bench_price = 0.080488

elif model_type == 'bs':
    if barr_type == 'barr':
        if opt_type == 'call':
            bench_price = 0.123493
        else: # put
            bench_price = 0.009535
    else: # van
        if opt_type == 'call':
            bench_price = 0.129944
        else: # put
            bench_price = 0.085942

In [ ]:
# CVAE training settings
dim_z       = 4 # 8, 12
hidden_dims = [128, 128, 64] # [128, 128, 64], [256, 256, 128], [512, 512, 256]
batch_size  = 4096 # 1024, 2048, 4096, 8192
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-4 # 1e-3, 1e-4, 1e-5, 1e-6
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 97
validation_chunk_idxs = [15,24,78]
#val_every_chunks = 10
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{warmup_chunks}_chunk{num_chunks}.pt"


n_samples = 10000 # n_samples= 1k, 10k, 100k
if n_samples % 2 != 0:
    raise ValueError("n_samples should be an even number for antithetic sampling")

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

# dim z = 2 distribution graph

In [ ]:
ckpts_no_bn = [
    f"result/cvae/{model_type}/cvae_{model_type}_2_{hidden_dims[0]}_{batch_size}_None_{lr}_{beta}_{validation_chunk_idxs}_chunk5.pt",
    f"result/cvae/{model_type}/cvae_{model_type}_2_{hidden_dims[0]}_{batch_size}_None_{lr}_{beta}_{validation_chunk_idxs}_chunk10.pt",
    f"result/cvae/{model_type}/cvae_{model_type}_2_{hidden_dims[0]}_{batch_size}_None_{lr}_{beta}_{validation_chunk_idxs}_chunk20.pt",
    f"result/cvae/{model_type}/cvae_{model_type}_2_{hidden_dims[0]}_{batch_size}_None_{lr}_{beta}_{validation_chunk_idxs}_chunk97.pt",
]

plot_z_white_fig4_style(
    checkpoint_paths=ckpts_no_bn,
    labels=["5%", "10%", "20%", "Complete"],
    model_type=model_type,
    chunk_idxs=validation_chunk_idxs,   # validation chunk index     
    seed=123,
    mode="sample",      # recognition sample z_q 사용
    normalize_x=False,  # input data norm 여부
    title="CVAE prior-standardized recognition latent, no BN",
    save_path="zwhite_no_bn.png",
    row_batch_size=2**16
)

In [ ]:
ckpts_bn = [
    f"result/cvae/{model_type}/cvae_{model_type}_2_{hidden_dims[0]}_{batch_size}_5_{lr}_{beta}_{validation_chunk_idxs}_chunk5.pt",
    f"result/cvae/{model_type}/cvae_{model_type}_2_{hidden_dims[0]}_{batch_size}_5_{lr}_{beta}_{validation_chunk_idxs}_chunk10.pt",
    f"result/cvae/{model_type}/cvae_{model_type}_2_{hidden_dims[0]}_{batch_size}_5_{lr}_{beta}_{validation_chunk_idxs}_chunk20.pt",
    f"result/cvae/{model_type}/cvae_{model_type}_2_{hidden_dims[0]}_{batch_size}_5_{lr}_{beta}_{validation_chunk_idxs}_chunk97.pt",
]

plot_z_white_fig4_style(
    checkpoint_paths=ckpts_bn,
    labels=["5%", "10%", "20%", "Complete"],
    model_type=model_type,
    chunk_idxs=validation_chunk_idxs,
    seed=123,
    mode="sample",
    normalize_x=False,
    title=f"CVAE prior-standardized recognition latent, {bn_chunks}% BN pretraining",
    save_path="zwhite_bn.png",
    row_batch_size=2**16
)